# Real-Time Face Mask Compliance Detection

To use the CDS6334 environment, run below conda command:

In [67]:
conda env export -n CDS6334 > CDS6334.yml


Note: you may need to restart the kernel to use updated packages.


In [68]:
# import libraries
# !pip install ultralytics tensorflow opencv-python-headless matplotlib seaborn scikit-learn pandas
import os
import shutil
import random
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning Imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from ultralytics import YOLO

print("Libraries installed and imported successfully.")

Libraries installed and imported successfully.


## 1.0 Data Loading

### 1.1 Define Paths

In [69]:
DATASET_PATH = r"C:\Users\Lee Hong Yi\Downloads\VIP Assignment\Real-Time-Face-Mask-Detection\Dataset"
class_dirs = ["with_mask", "without_mask", "mask_weared_incorrect"]
print("DATASET_PATH exists:", os.path.exists(DATASET_PATH))
for d in class_dirs:
    p = os.path.join(DATASET_PATH, d)
    print(d, "exists:", os.path.exists(p), "num_images:", len(glob(os.path.join(p, "*.*"))))

DATASET_PATH exists: True
with_mask exists: True num_images: 2994
without_mask exists: True num_images: 2994
mask_weared_incorrect exists: True num_images: 2994


## 2.0 Data Preprocessing

### 2.2 Preprocessing for Model B (EfficientNet Format)

#### 2.2.1 Create Crop Directories

In [70]:
# Creates 3 folders: crops/with_mask, crops/without_mask, crops/incorrect.
# Ensure crop directories exist
CROPS_PATH = os.path.join(DATASET_PATH, "crops")
CROP_CLASSES = ["with_mask", "without_mask", "mask_weared_incorrect"]

for c in CROP_CLASSES:
    os.makedirs(os.path.join(CROPS_PATH, c), exist_ok=True)

def is_image_valid(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None or img.size == 0:
            return False
        return True
    except Exception:
        return False

#### 2.2.2 Crop and Sort

In [71]:


# Logic: It loops through every image, looks at the bounding box, cuts only the face out of the image, 
# and saves that tiny face image into the correct folder.

# Why: EfficientNet is a classifier. It needs to look at just the face to decide if the mask is 
# correct or not.
def augment_image(img):
    aug_imgs = [img]
    if img is not None and img.size > 0:
        # Horizontal flip
        aug_imgs.append(cv2.flip(img, 1))
        # Rotate 15 degrees
        M = cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), 15, 1)
        aug_imgs.append(cv2.warpAffine(img, M, (img.shape[1], img.shape[0])))
        # Brightness adjustment
        aug_imgs.append(cv2.convertScaleAbs(img, alpha=1.2, beta=30))
    return aug_imgs

# Loop through all PNG images in each class folder
for c in CROP_CLASSES:
    img_dir = os.path.join(DATASET_PATH, c)
    img_files = glob(os.path.join(img_dir, "*.png"))
    for img_path in tqdm(img_files, desc=f"Processing {c}"):
        if not is_image_valid(img_path):
            continue  # Skip corrupted images
        img = cv2.imread(img_path)
        # If bounding box info is available, crop face here (not implemented, so use full image)
        face_img = img  # Placeholder for cropped face
        for idx, aug in enumerate(augment_image(face_img)):
            save_path = os.path.join(CROPS_PATH, c, f"{os.path.splitext(os.path.basename(img_path))[0]}_{idx}.png")
            cv2.imwrite(save_path, aug)

Processing with_mask:   0%|          | 0/2994 [00:00<?, ?it/s]

Processing with_mask:   0%|          | 0/2994 [00:00<?, ?it/s]

Processing mask_weared_incorrect: 100%|██████████| 2994/2994 [00:19<00:00, 155.58it/s]


In [72]:
# Optional: Preprocess to brighten and sharpen dim or blurry images before augmentation
# This will brighten and sharpen all images before saving to crops directory

def brighten_image(img, alpha=1.0, beta=40):
    # alpha: contrast (1.0-3.0), beta: brightness (0-100)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def sharpen_image(img):
    kernel = np.array([[0, -1, 0],
                      [-1, 5, -1],
                      [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

for c in CROP_CLASSES:
    img_dir = os.path.join(DATASET_PATH, c)
    img_files = glob(os.path.join(img_dir, "*.png"))
    for img_path in tqdm(img_files, desc=f"Brightening & Sharpening {c}"):
        if not is_image_valid(img_path):
            continue
        img = cv2.imread(img_path)
        bright_img = brighten_image(img, alpha=1.0, beta=40)  # Adjust beta as needed
        sharp_img = sharpen_image(bright_img)
        # Use sharp_img for augmentation and saving
        face_img = sharp_img  # If bounding box, crop here
        for idx, aug in enumerate(augment_image(face_img)):
            save_path = os.path.join(CROPS_PATH, c, f"{os.path.splitext(os.path.basename(img_path))[0]}_{idx}.png")
            cv2.imwrite(save_path, aug)


Brightening & Sharpening mask_weared_incorrect: 100%|██████████| 2994/2994 [00:18<00:00, 163.96it/s]


#### 2.2.3 Data Generators

In [ ]:
# Sets up ImageDataGenerator which automatically loads these cropped images in batches and applies 
# "Augmentation" (randomly rotating them slightly) to make the model smarter.
# Paths
train_dir = os.path.join(CROPS_PATH)  # If you have a split, use subfolders like 'train', 'test'

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
print(f"[INFO] Training generator: Found {train_gen.samples} images belonging to {train_gen.num_classes} classes. These will be used for training.")

test_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)
print(f"[INFO] Testing generator: Found {test_gen.samples} images belonging to {test_gen.num_classes} classes. These will be used for Testing.")

Found 28743 images belonging to 3 classes.
[INFO] Training generator: Found 28743 images belonging to 3 classes. These will be used for training.
Found 7185 images belonging to 3 classes.
[INFO] Testing generator: Found 7185 images belonging to 3 classes. These will be used for validation.


## 3.0 Model B: Two-Stage Classifier (EfficientNet-B0)

### 3.1 Build Architecture

In [97]:
# Modifies the standard EfficientNet.

# load EfficientNet but cut off the "Head" (the top layer). 
# You replace it with your own "3-Class Output Layer" so it predicts your specific mask classes instead of generic objects.

### 3.2 Compile Model

In [98]:
# Configures the learning rules.

# Sets the optimizer to Adam (standard for vision) and loss to CategoricalCrossentropy 
# (standard for multi-class classification).

### 3.3 Training

In [99]:
# Trains the classifier on the cropped faces from Step 2.2.

### 3.4 Evaluation (EfficientNet)

In [100]:
# checks accuracy specifically on the cropped faces.

# Generates a classification report showing Precision, Recall, and F1-score for each of the 3 classes.

## 5.0 Comparative Analysis

### 5.1 Metrics Comparison

In [101]:
# Puts the results side-by-side.

# create a table or bar chart comparing the Recall for 'Incorrect Mask'. 
# Question answered: Which model is less likely to miss a bad mask?

### 5.2 Inference Speed Test (FPS)

In [102]:
# Measures "Real-World" speed.

# Run YOLO on 100 images -> Measure time.
# Run Face Detect + EfficientNet on 100 images -> Measure time.
# Result: Show that YOLO is likely faster, but check if EfficientNet is "fast enough."

## 6.0 Application Simulation (Access Control)